In [1]:
from GNN import GATv2SequenceModel
from dataset_processing import DatasetProcessing
import json
import networkx as nx
from torch_geometric.utils import to_networkx
import random
import torch
from torch.utils.data import DataLoader

In [ ]:
dataset_process=DatasetProcessing()

with open("data_savt.json","r") as f:
    test_set=json.load(f)
a,label_list=dataset_process.modified(test_set)
sequence_list,sequence_labels=dataset_process.create_sequence_and_labels(a,label_list,5)

2

In [9]:
dataset=[]
for graphs,labels in zip(sequence_list,sequence_labels):
    dataset.append((graphs,labels[0],labels[1]))

In [10]:
def collate_fn(batch):
    """
    Custom collate function for handling sequences of graphs.
    Input:
        batch: List of tuples (sequence of graphs, label)
    Output:
        sequences: List of sequences (each sequence is a list of graphs)
        labels: Tensor of labels
    """
    sequences, labels, bool_labels = zip(*batch)
    return (
        sequences,
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(bool_labels, dtype=torch.float),
    )


train_loader = DataLoader(
    dataset=dataset, batch_size=1, shuffle=True, collate_fn=collate_fn
)

In [ ]:
in_channels = 13
hidden_channels = 16
out_channels = 10
edge_dim = 3
heads = 1
num_epochs = 100
batch_size = 5
learning_rate = 0.001



model = GATv2SequenceModel(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    edge_dim=edge_dim,
    out_channels=out_channels,
    heads=heads,
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
criterion_bools=torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for sequences, labels, bool_labels in train_loader:
        # Move data to GPU if available
        sequences=sequences[0]
        for i in sequences:
            i.to("cuda")
        labels = labels.to("cuda")
        bool_labels=bool_labels.to("cuda")
        outputs,bools = model(sequences)

        # Compute loss
        loss_class = criterion(outputs, labels)
        loss_bool =criterion_bools(bools,bool_labels)
        loss=loss_class+loss_bool

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/100, Loss: 3.0923
Epoch 2/100, Loss: 2.8289
Epoch 3/100, Loss: 2.6009
Epoch 4/100, Loss: 2.6551
Epoch 5/100, Loss: 2.4890
Epoch 6/100, Loss: 2.3599
Epoch 7/100, Loss: 2.2844
Epoch 8/100, Loss: 2.2075
Epoch 9/100, Loss: 2.1138
Epoch 10/100, Loss: 1.9922
Epoch 11/100, Loss: 1.9483
Epoch 12/100, Loss: 1.8569
Epoch 13/100, Loss: 1.8082
Epoch 14/100, Loss: 1.7804
Epoch 15/100, Loss: 1.8067
Epoch 16/100, Loss: 1.6364
Epoch 17/100, Loss: 1.6124
Epoch 18/100, Loss: 1.6473
Epoch 19/100, Loss: 1.5328
Epoch 20/100, Loss: 1.5106
Epoch 21/100, Loss: 1.4581
Epoch 22/100, Loss: 1.5170
Epoch 23/100, Loss: 1.4165
Epoch 24/100, Loss: 1.3962
Epoch 25/100, Loss: 1.4293
Epoch 26/100, Loss: 1.3803
Epoch 27/100, Loss: 1.2661
Epoch 28/100, Loss: 1.2209
Epoch 29/100, Loss: 1.2084
Epoch 30/100, Loss: 1.2244
Epoch 31/100, Loss: 1.2682
Epoch 32/100, Loss: 1.2537
Epoch 33/100, Loss: 1.2013
Epoch 34/100, Loss: 1.1165
Epoch 35/100, Loss: 1.1408
Epoch 36/100, Loss: 1.1175
Epoch 37/100, Loss: 1.1418
Epoch 38/1

In [13]:
torch.save(model.state_dict(),"trained.pth")

In [18]:
model.eval()
model(sequence_list[0])

(tensor([[-0.7633, -3.6663,  1.9229,  1.2298, -3.7150, -3.5651,  1.1065, -2.6405,
          -3.2948, -3.1038]], device='cuda:0', grad_fn=<AddmmBackward0>),
 tensor([[-3.8294, -3.9998]], device='cuda:0', grad_fn=<AddmmBackward0>))

In [19]:
label_list[0]

(2, [0, 0])